1. Identification of the top 5 best-selling items.
2. Calculation of the average number of items per order.
3. Determination of the average price per item.
4. Examination of the relationship between average price per item and average quantity sold per item.

In [ ]:
# read bigquery data into pandas dataframe
import pandas as pd


df = pd.read_gbq(
    """
    SELECT  *
    FROM `jr-data-training.cafe.cafe-sales`
    --- LIMIT 10
  """,
    project_id="jr-data-training",
    location="australia-southeast1",
)

In [ ]:
#1. Identification of the top 5 best-selling items.
import json
from collections import Counter
def parse_items(items_string):
    try:
        order_data = json.loads(items_string)  
        cart_items = order_data['cart'] 
        # Extract the item
        return [(item['name'], item['price']) for item in cart_items if isinstance(item, dict)]
    except (json.JSONDecodeError, TypeError):
        return None

# change the item json to item list
df['parsed_items'] = df['items'].apply(parse_items)

# give the item list
all_items = df['parsed_items'].apply(lambda items: [item[0] for item in items] if items is not None else [])
all_items =  [item for sublist in all_items for item in sublist]

# Count each product
item_counter = Counter(all_items)

top_5_items = item_counter.most_common(5)

top_5_df = pd.DataFrame(top_5_items, columns=['Item', 'Sales'])

print("Top 5 Best-Selling Items:")
print(top_5_df)


In [ ]:
#2. Calculation of the average number of items per order.
def items_num(items_string):
    try:
        order_data = json.loads(items_string)  
        cart_items = order_data['cart_size'] 
        # Extract the item
        return cart_items
    except (json.JSONDecodeError, TypeError):
        return None
df['items_num'] = df['items'].apply(items_num)
avg_items_per_order =df['items_num'].mean()
print(f"Average Number of Items per Order: {avg_items_per_order}")

In [ ]:
#3. Determination of the average price per item.
df['price_per_item'] = df['total'] / df['items_num']
avg_price_per_item = df['price_per_item'].mean()

print(f"Average Price per Item: {avg_price_per_item}")

In [ ]:
#4. Print the item price and quantity sold, plot the relationship between average price per item and average quantity sold per item.
import matplotlib.pyplot as plt
from collections import defaultdict

# items_details = Counter([item for sublist in df['parsed_items'] for item in sublist])
item_price_count = defaultdict(lambda: {'total_price': 0, 'count': 0})


for items in df['parsed_items']:
    if items:  # Only items are not none
        for item, price in items:
            item_price_count[item]['total_price'] += price
            item_price_count[item]['count'] += 1

# Transfer to dataframe
item_stats = pd.DataFrame(item_price_count).T.reset_index()
item_stats.columns = ['item', 'total_price', 'count']

# Calculate the avg price
item_stats['avg_price'] = item_stats['total_price'] / item_stats['count']

print(item_stats)

# Plot the Scatter Graph, find the number of item sold based on the price
plt.figure(figsize=(10, 6))  
plt.scatter(item_stats['avg_price'], item_stats['count'])
plt.xlabel('avg_price')
plt.ylabel('item num')
plt.title('avg_pric and item num')
plt.show()
